# 範例 1：FastQC + MultiQC 定序資料品質管控（QC）入門

這是「[class](../README.md)」課程資料夾的第一個實作範例，取代原本的 `第一章_程式碼_文字接龍.ipynb`。

**為什麼把 QC 放在第一個範例？** 任何定序分析（QIIME2 16S、RNA-seq、WGS…）開始前，
第一步永遠是檢查原始資料品質。這個範例會：

1. 用 Python 產生 6 個「模擬」FASTQ 樣本，刻意做出不同的品質問題（品質下降、有 adapter、重複序列過多、GC 含量偏移、整體低品質）
2. 用 **FastQC** 對每個樣本個別跑品質報告
3. 用 **MultiQC** 把所有樣本的 FastQC 結果彙整成一份總覽報告
4. 直接在 notebook 裡看報告，練習判讀「哪個樣本有問題、問題出在哪」

> 這台機器（台灣杉三號 GP1 生醫節點）已經有現成的 `FastQC/0.11.9`、`MultiQC/1.18` 模組，
> 不需要額外安裝，也不依賴外部網路連線，很適合當作環境設定好之後的第一個驗證練習。

對應教學文件：[01_FastQC_MultiQC_教學.md](01_FastQC_MultiQC_教學.md)


## Step 1：產生模擬 FASTQ 資料

真實定序資料下載費時，這裡改用 Python 直接產生 6 個模擬樣本，每個樣本刻意設計成
展示 FastQC 報告中一種常見的品質問題，方便對照學習：

| 樣本 | 設計的問題 | 預期在報告上看到 |
|---|---|---|
| `sample_A_good` | 無（對照組，品質正常） | 大部分模組都是綠色（PASS） |
| `sample_B_degrading` | 讀長後段（3' 端）品質逐漸下降 | "Per base sequence quality" 出現黃色/紅色 |
| `sample_C_adapter` | 部分 reads 尾端接上固定 adapter 序列 | "Overrepresented sequences" / "Adapter Content" 出現警告 |
| `sample_D_duplicates` | 大量重複序列 | "Sequence Duplication Levels" 出現紅色 |
| `sample_E_gc_skew` | 混合兩群不同 GC bias（雙峰分布） | "Per sequence GC content" 出現紅色 |
| `sample_F_lowqual` | 整體 Phred 品質偏低（約 Q10~15） | "Per base sequence quality" 整體偏黃/紅 |


In [ ]:
import random
import gzip
import os

random.seed(42)

WORKDIR = "/work/c00cjz00/notebook/class/01_fastqc_multiqc/fastq"
os.makedirs(WORKDIR, exist_ok=True)

BASES = "ACGT"
READ_LEN = 150
N_READS = 2000
ADAPTER = "AGATCGGAAGAGC"  # Illumina TruSeq adapter 常見序列片段


def random_seq(length, gc_bias=0.5):
    seq = []
    for _ in range(length):
        if random.random() < gc_bias:
            seq.append(random.choice("GC"))
        else:
            seq.append(random.choice("AT"))
    return "".join(seq)


def phred_to_char(q):
    q = max(2, min(41, q))
    return chr(33 + q)


def quality_string(length, base_q=36, degrade=False, lowqual=False):
    quals = []
    for i in range(length):
        q = base_q
        if lowqual:
            q = random.randint(8, 16)
        elif degrade:
            # 品質從第 90 個 base 開始線性下降
            drop_start = int(length * 0.6)
            if i > drop_start:
                q = base_q - int((i - drop_start) / (length - drop_start) * (base_q - 8))
        q += random.randint(-2, 2)
        # 每個 base 有 2% 機率掉到低品質（Q10~20）：真實定序資料本來就會有零星低品質 base，
        # 這裡刻意加入是為了避免全部品質字元都 >=64，導致 FastQC 誤判成 Phred+64 編碼（見下方說明）
        if not lowqual and random.random() < 0.02:
            q = random.randint(10, 20)
        quals.append(phred_to_char(q))
    return "".join(quals)


def write_fastq(path, records):
    with gzip.open(path, "wt") as f:
        for read_id, seq, qual in records:
            f.write(f"@{read_id}\n{seq}\n+\n{qual}\n")


def make_sample(name, n_reads=N_READS, gc_bias=0.5, gc_bias_mix=None, degrade=False,
                 lowqual=False, add_adapter=False, dup_rate=0.0):
    records = []
    pool_size = max(1, int(n_reads * (1 - dup_rate)))

    if gc_bias_mix:
        # 雙峰 GC 分布：混合兩群不同 GC bias 的 reads，觸發 FastQC 的
        # "Per Sequence GC Content" 模組（單一群體整批偏移不會觸發，見下方說明）
        template_pool = []
        per_group = max(1, pool_size // len(gc_bias_mix))
        for bias in gc_bias_mix:
            template_pool += [random_seq(READ_LEN, bias) for _ in range(per_group)]
    else:
        template_pool = [random_seq(READ_LEN, gc_bias) for _ in range(pool_size)]

    for i in range(n_reads):
        if dup_rate > 0 and random.random() < dup_rate:
            seq = random.choice(template_pool[: max(1, len(template_pool) // 20)])
        else:
            # 依序取用 template_pool（而非 random.choice 重複抽樣），確保「非刻意重複」的樣本
            # 不會因為抽樣重複而意外觸發 Sequence Duplication Levels 警告
            seq = template_pool[i % len(template_pool)]

        if add_adapter and random.random() < 0.35:
            cut = random.randint(80, READ_LEN - len(ADAPTER) - 1)
            seq = seq[:cut] + ADAPTER + seq[cut:]
            seq = seq[:READ_LEN]

        qual = quality_string(len(seq), degrade=degrade, lowqual=lowqual)
        records.append((f"{name}_read{i}", seq, qual))

    out_path = os.path.join(WORKDIR, f"{name}.fastq.gz")
    write_fastq(out_path, records)
    print(f"寫入 {out_path}（{n_reads} reads）")


make_sample("sample_A_good", gc_bias=0.5)
make_sample("sample_B_degrading", gc_bias=0.5, degrade=True)
make_sample("sample_C_adapter", gc_bias=0.5, add_adapter=True)
make_sample("sample_D_duplicates", gc_bias=0.5, dup_rate=0.7)
make_sample("sample_E_gc_skew", gc_bias_mix=[0.85, 0.15])
make_sample("sample_F_lowqual", gc_bias=0.5, lowqual=True)

print("\n完成，檔案清單：")
for f in sorted(os.listdir(WORKDIR)):
    print(" ", f)


> **真實踩坑案例**：這份模擬資料產生器原本有兩個 bug，是另一位使用者實際把整條 pipeline 跑過一輪後才發現的
> （不是憑空發現，是真的送 SLURM job 跑出來、逐行比對 `fastqc_data.txt` 才抓到）：
>
> 1. **Phred+64 編碼誤判**：原本的品質分數都壓在 Q34–38，換算成 ASCII 全部 ≥64。
>    FastQC 判斷編碼的邏輯是「如果整個檔案裡的品質字元全部 ≥64，就假設是舊式的 Illumina 1.5（Phred+64）」——
>    結果原本的 Q35 被當成 Phred+64 解讀成 Q3~7，害對照組 `sample_A_good` 的品質模組全部顯示 FAIL。
>    **解法**：讓每個 base 有 2% 機率掉到 Q10~20（低於 64 的門檻），這樣 FastQC 就能正確判斷成 Sanger/Illumina 1.9
>    （也就是標準 Phred+33），順便讓資料更接近真實定序資料本來就會有的零星低品質 base。
> 2. **GC skew 設計方式沒觸發到對應模組**：原本 `sample_E_gc_skew` 是整批 reads 都用同一個偏高的 GC bias 產生，
>    但 FastQC 的 "Per Sequence GC Content" 模組是拿「觀測分布」跟「以觀測平均值為中心建出來的常態分布」比較，
>    整批一起偏移並不會被抓到（因為觀測平均值本身也跟著偏移，兩者仍然吻合）。
>    **解法**：改成混合兩群不同 GC bias 的 reads（85% 與 15%），產生雙峰分布，才會真的偏離常態分布形狀而觸發警告。


## Step 2：對每個樣本跑 FastQC

FastQC 會對每個 `.fastq.gz` 檔案產生一份 HTML 報告，裡面有十幾個獨立模組
（Per base sequence quality、Per sequence GC content、Sequence Duplication Levels…），
每個模組會標示 <span style="color:green">✅ PASS</span> /
<span style="color:orange">⚠️ WARN</span> / <span style="color:red">❌ FAIL</span>。

這裡直接呼叫模組安裝的完整路徑（避免又踩到 `%%bash` PATH 沒繼承 `module load` 的問題，
原理跟我們在 QIIME2 notebook 遇到的 PATH 議題一樣）。


In [ ]:
%%bash
FASTQC=/opt/ohpc/Taiwania3/pkg/biology/FastQC/FastQC_v0.11.9/fastqc
WORKDIR=/work/c00cjz00/notebook/class/01_fastqc_multiqc

mkdir -p "${WORKDIR}/fastqc_out"
"${FASTQC}" --outdir "${WORKDIR}/fastqc_out" "${WORKDIR}"/fastq/*.fastq.gz

echo "=== fastqc_out 內容 ==="
ls "${WORKDIR}/fastqc_out"


## Step 3：用 MultiQC 彙整成一份總覽報告

6 個樣本各自打開 FastQC 報告太花時間，**MultiQC** 會掃描一個資料夾裡所有工具的輸出，
自動彙整成一份互動式總覽報告，一眼比較所有樣本的品質差異。


In [ ]:
%%bash
MULTIQC=/opt/ohpc/Taiwania3/pkg/biology/MultiQC/MultiQC_v1.18/bin/multiqc
WORKDIR=/work/c00cjz00/notebook/class/01_fastqc_multiqc

cd "${WORKDIR}"
"${MULTIQC}" fastqc_out --outdir multiqc_out --force

echo "=== multiqc_out 內容 ==="
ls multiqc_out


## Step 4：直接在 Notebook 裡看報告

不用下載檔案或另外開瀏覽器，直接把 MultiQC 產生的 HTML 報告嵌入 notebook 顯示：


In [ ]:
> 若 IFrame 在你的 VS Code 環境顯示不出來，也可以直接在檔案總管開啟
> `class/01_fastqc_multiqc/multiqc_out/multiqc_report.html`，VS Code 會用內建瀏覽器預覽。

---

## 練習題

1. 打開 MultiQC 總覽報告，找出哪個樣本在 **"Sequence Duplication Levels"** 是紅色（FAIL）？
   跟我們產生資料時的哪個參數設計對應得上？
2. `sample_E_gc_skew` 在 **"Per Sequence GC Content"** 模組會出現什麼樣的分布圖形？
   為什麼要混合兩群不同 GC bias（雙峰分布）才會讓這個模組觸發警告，而單純整批調高 `gc_bias` 不會？
   （提示：見 Step 1 程式碼上方的「真實踩坑案例」說明）
3. 如果這是真實定序資料，`sample_C_adapter` 這種 adapter 污染問題，
   在正式分析前應該用什麼工具、下一步怎麼處理？（提示：對照 QIIME2 pipeline 裡的
   `qiime demux`/`qiime quality-filter` 之前，通常會先做 adapter trimming）
4. 試著修改 Step 1 的 `make_sample(...)` 參數（例如把 `dup_rate` 調到 0.95），
   重新執行整個 notebook，觀察 MultiQC 報告如何反映你的改動。
5. `quality_string()` 裡有一段刻意讓每個 base 有 2% 機率掉到低品質的邏輯，為什麼拿掉它會讓
   FastQC 誤判編碼格式？試著把這行拿掉重新產生 `sample_A_good`，實際觀察 `fastqc_data.txt`
   裡的 `Encoding` 欄位變化。


> 若 IFrame 在你的 VS Code 環境顯示不出來，也可以直接在檔案總管開啟
> `class/01_fastqc_multiqc/multiqc_out/multiqc_report.html`，VS Code 會用內建瀏覽器預覽。

---

## 練習題

1. 打開 MultiQC 總覽報告，找出哪個樣本在 **"Sequence Duplication Levels"** 是紅色（FAIL）？
   跟我們產生資料時的哪個參數設計對應得上？
2. `sample_E_gc_skew` 在 **"Per Sequence GC Content"** 模組會出現什麼樣的分布圖形？
   為什麼刻意調高 `gc_bias` 會讓這個模組出現警告？
3. 如果這是真實定序資料，`sample_C_adapter` 這種 adapter 污染問題，
   在正式分析前應該用什麼工具、下一步怎麼處理？（提示：對照 QIIME2 pipeline 裡的
   `qiime demux`/`qiime quality-filter` 之前，通常會先做 adapter trimming）
4. 試著修改 Step 1 的 `make_sample(...)` 參數（例如把 `dup_rate` 調到 0.95），
   重新執行整個 notebook，觀察 MultiQC 報告如何反映你的改動。
